In [3]:
import functools
import pandas as pd
import sys
sys.path.append('../python_scripts')
import xarray as xr

from utils import get_pareto_layers

from objective_functions import mean_annualized_return, WeightedRegimeApplyer, weighted_mean, extract_fold_regimes

df_all = pd.DataFrame()
df_regimes = pd.DataFrame()
for fold in range(4):
    run_name = '2d_test_fold_{}'.format(fold)
    df = pd.read_parquet('../sim_results/{}/median_objectives.parquet'.format(run_name)).reset_index()

    


    df_1  = pd.DataFrame()
    for gen in df['gen'].drop_duplicates():
        df_gen = df.loc[df['gen'] == gen]
        df_pivot = pd.pivot_table(df_gen[['mode', 'objective', 'sim_id', 'value']], index = 'sim_id', values = 'value', columns = ['mode', 'objective'])
        df_layers = get_pareto_layers(df_pivot[[('train','mean_regret'), ('train','regret_quantile')]], sense = ['min', 'min'], num_layers=1)
        pareto_indices = df_layers.loc[df_layers.layer==0].index
        df_add = df_pivot.loc[pareto_indices]
        df_add['gen'] = gen
        argmin = int(df_add[('train', 'mean_regret')].argmin())
        df_add_1 = pd.DataFrame(df_add.iloc[argmin]).transpose() 
        df_1 = pd.concat([df_1, df_add_1])


    s_voo = xr.open_dataset('../simulation_data/momentum.nc').to_array()[0].sel(symbol = 'VOO', band = 'price_end').to_pandas()
    df_folds = pd.read_parquet('../strategy/folds.parquet')
    df_fold = df_folds.loc[df_folds.fold_index == fold]

    start_date = min(df_fold.start_date)
    end_date = max(df_fold.end_date)
    agg_func = functools.partial(mean_annualized_return, start_date, end_date)

    mean_func = WeightedRegimeApplyer(df_fold, agg_func, weighted_mean)
    voo_mean = mean_func(s_voo)

    sim_id = df_1.index[-1]

    s3_path = "s3://jdinvestment/{}/portfolio_values/sim_{}.parquet".format(run_name, sim_id)
    if run_name == '2d_test_fold_0':
        s3_path = s3_path.replace('2d_test_fold_0', '2d_test_4')
    s_values = pd.read_parquet(s3_path).transpose().iloc[:-2,0]
    s_values.index = pd.to_datetime(s_values.index.str.replace('_',' '))
    opt_mean = mean_func(s_values)

    df_add_regimes = extract_fold_regimes(df_folds.loc[df_folds.fold_index == fold], s_values)

   

    print(fold, sim_id, voo_mean, opt_mean)


0 25cb957e6dab 0.05915473235413306 0.165488685965797


KeyError: Timestamp('2007-12-24 00:00:00')

In [4]:
1+1

2

In [3]:
df_add_regimes

(                    value  regime_index
 date                                   
 2009-04-13    447702.5516             0
 2009-05-11    450092.8439             0
 2009-06-08    470807.4854             0
 2009-07-06    459955.2221             0
 2009-08-03    503512.3163             0
 2009-08-31    507303.8208             0
 2009-09-28    509086.2408             0
 2009-10-26    511236.0133             0
 2009-11-23    515901.3243             0
 2009-12-21    518534.7137             0
 2011-05-09    734946.2652             1
 2011-06-06     691058.949             1
 2011-07-04    713509.2431             1
 2011-08-01    690152.0483             1
 2011-08-29     640263.637             1
 2011-09-26    621228.0184             1
 2011-10-24    658862.5872             1
 2011-11-21    629893.0666             1
 2016-03-07   1512745.8709             2
 2016-04-04   1561387.5406             2
 2016-05-02   1590667.3145             2
 2016-05-30     1591379.04             2
 2016-06-27   15

In [ ]:

import numpy as np
df_regimes = df_regimes.sort_index()
df_partition = df_regimes[['fold', 'regime_index']].drop_duplicates()
df_regimes = df_regimes.set_index(['fold', 'regime_index'])
for fold, regime_index in df_partition.values:
    df_last = pd.DataFrame(df_regimes.loc[(last_fold, last_regime_index)].iloc[-1])
    df1 = df_regimes.loc[(fold, regime_index)]

    last_fold, last_regime_index = fold, regime_index

    



{np.int64(7), np.int64(15), np.int64(213), np.int64(215), np.int64(42), np.int64(191)}
{np.int64(68), np.int64(156), np.int64(98)}


In [15]:
df_all

date,2008-01-21,2008-02-18,2008-03-17,2008-04-14,2008-05-12,2008-06-09,2008-07-07,2008-08-04,2008-09-01,2008-09-29,...,2025-06-16,2025-07-14,2025-08-11,2025-09-08,2025-10-06,2025-11-03,2025-12-01,2025-12-29,2026-01-26,2026-02-23
25cb957e6dab,407839.0,410229.3308,399671.9475,419967.2686,437798.5242,434014.1924,430984.3938,458597.9938,468509.7643,456297.2548,...,15526930.3911,15769220.1279,15754561.2498,15920682.205,15899205.3034,15943563.561,16129867.3851,16379053.4622,16328765.9986,16131116.6716
d32bb00f566f,407986.0,406581.185,403515.9788,406446.7058,411798.1218,408244.4396,407121.624,405338.5067,406273.4379,393681.6979,...,10433788.2261,10478486.3761,10389227.6864,10727209.5242,10816021.5219,10692507.5603,10883071.1149,10866681.9841,11229384.5088,11145368.888
a99516312234,407846.0,407928.9455,404740.2937,421761.8007,427320.1339,427117.4283,426037.78,421293.5084,423241.8433,403930.035,...,15951437.6774,16115187.493,16184363.1987,16642466.9921,16796739.2184,16442056.8072,16796194.3502,16805973.1556,17390076.7375,17161683.5259
2e1ebcb6dc55,407888.0,408347.7956,388446.6293,403315.2816,415970.2571,411199.0724,405137.5535,421729.0424,423472.8926,381720.3344,...,19623681.6297,19900632.859,19882209.8376,20043474.1086,20137206.488,19966229.5005,20457053.5631,20847233.1965,21210557.0721,20953798.9559
